In [ ]:
def f(x):
    if x <= 2:
        return x**2 - 1
    elif x <= 100:
        return x*0 + 3
    else:
        return x - 97

def g(x):
    if x <= 5:
        return x**2 - 4
    elif x <= 100:
        return x*0 + 16
    else:
        return x - 84

In [ ]:
from matplotlib import pyplot as plt
plot_range = range(-3, 105)
plt.plot(plot_range, [f(x) for x in plot_range], label='f(x)')
plt.plot(plot_range, [g(x) for x in plot_range], label='g(x)')
plt.vlines(x=-2, ymin=-5, ymax=20, color='orange', linewidth=0.5, ls='--')
plt.vlines(x=2, ymin=-5, ymax=20, color='orange', linewidth=0.5, ls='--')
plt.legend()

In [ ]:
import torch
from humancompatible.train.dual_optim import ALM, PBM

INIT_X = 2.1

x = torch.nn.Parameter(torch.tensor(INIT_X, dtype=torch.float32))
opt = torch.optim.SGD([x], lr=0.1)
# dual = ALM(m=1, lr=0.01, is_ineq=True, augmentation='hpr', penalty=0.1)
dual = PBM(m=1, gamma=0., gamma_annealing=False, penalty_annealing=False)

rng = torch.random.seed()

xs = []


for i in range(100):
    y = f(x) + torch.rand(1) *5
    c = g(x) + torch.rand(1) *5
    lagr = dual.forward_update(y, c)
    lagr.backward()
    print(f"Iteration {i}: x = {x.item()}, f(x) = {y.item()}, g(x) = {c.item()}, L_grad = {x.grad.item()}")
    xs.append(x.item())

    opt.step()
    opt.zero_grad()

In [ ]:
from matplotlib import pyplot as plt
plot_range = range(-5, int(INIT_X) + 5)
plt.plot(plot_range, [f(x) for x in plot_range], label='f(x)')
plt.plot(plot_range, [g(x) for x in plot_range], label='g(x)')
plt.vlines(x=-2, ymin=-5, ymax=20, color='orange', linewidth=0.5, ls='--')
plt.vlines(x=2, ymin=-5, ymax=20, color='orange', linewidth=0.5, ls='--')
plt.hlines(y=0, xmin=-5, xmax=10, color='orange', linewidth=0.5, ls='--')
plt.scatter(xs, [f(x) for x in xs], color='red', label='Optimization Path')
plt.legend()